# SDR VEP benchmark

Benchmarks the variant effect predictions of pretrained AlphaGenome (cell-type-matched CM/FB tracks and all
heart tracks), fine-tuned AlphaGenome, fine-tuned Borzoi and ChromBPNet against the measured effects of the
SDR-seq variants (SDR007AB, SDR010ABC; CM and FB):

1. **Effect size**: predicted RNA score vs measured log2FC, stratified by TSS distance and annotation.
2. **Effect sign**: does the sign of the RNA score predict the direction of the measured effect
   (positive variants; AUROC/AUPRC, contingency tables at 0 and at the F1-optimal cutoff)?
3. **Causality**: do absolute scores separate variants with (`var_type == "positive"`) and without a
   measured effect (AUPRC, Mann-Whitney U box plots, bootstrap power analysis), per model and across models?
4. **Richter-prioritised variants**: do the CHD-enriched Richter de novo variants measured in SDR007
   affect more genes than the other SDR007 variants?

**Inputs** (`config/paths.yaml`): `VCF_SDR007AB_DLI_SDR010ABC_Comb_Meta.csv` in `variant_data_dir`
(written by `variant_datasets/sdr_variant_dataset.ipynb`); the VEP outputs of the workflows in `vep_dir`
(`alphagenome_api/04_vep.sh`, `alphagenome_ft/04_vep.sh`, `borzoi/04_vep.sh`, `chrombpnet/03_vep.sh`, all run
on `VCF_SDR007AB_DLI_SDR010ABC_Comb.csv`); the SDR007 MAST tables (`sdr007_de_dir`); the gene annotation
(`gtf`, strand matching of the fine-tuned AlphaGenome RNA tracks). Saving figures (to
`figures_dir/sdr_vep_benchmark`) is commented out.

In [ ]:
import numpy as np
import pandas as pd

from cardiomics_s2f.config import load_paths
from cardiomics_s2f.eval.track_prediction import get_gtf
from cardiomics_s2f.vep.plotting import (
    box_grid,
    plot_AUPRCs,
    plot_AUROCs,
    plot_contingency_grid,
    plot_scatter_panel,
    plot_stratified_auprc_heatmap,
)
from cardiomics_s2f.vep.processing import (
    add_abs_score_columns,
    create_tss_stratifications,
    format_alphagenome_ft_vep_per_gene,
    format_borzoi_vep_per_gene,
    format_chrombpnet_vep,
    pivot_tf_scores,
    strip_phasing_suffix,
)
from cardiomics_s2f.vep.stats import mannwhitney_power_grid

# Prepare Data

In [ ]:
paths = load_paths()
VCF_NAME = "VCF_SDR007AB_DLI_SDR010ABC_Comb"   # VEP input written by sdr_variant_dataset.ipynb
SDR_VCF_META_FILE = paths.variant_data_dir / f"{VCF_NAME}_Meta.csv"
SDR007_FILE_PATH = paths.sdr007_de_dir         # raw MAST tables (Richter-prioritised variants)
FIGURE_PATH = f"{paths.figures_dir}/sdr_vep_benchmark/"
VEP_DIR = paths.vep_dir

#pretrained AG (workflows/alphagenome_api/04_vep.sh; groups from track_aggregations.yaml)
AG_pre_CM_SDR_VEP = VEP_DIR / f"AG_VEP_out/{VCF_NAME}_CM_and_earlyCM_tracks_aggr.csv"
AG_pre_FB_SDR_VEP = VEP_DIR / f"AG_VEP_out/{VCF_NAME}_Fibroblast_tracks_aggr.csv"
AG_pre_HEART_SDR_VEP = VEP_DIR / f"AG_VEP_out/{VCF_NAME}_naive_heart_tracks_aggr.csv"

#chrombpnet (workflows/chrombpnet/03_vep.sh, mean over folds)
cBPN_CM_SDR_VEP = VEP_DIR / f"chromBPNet_VEP_out/{VCF_NAME}_Cardiomyocytes.mean.variant_scores.tsv"
cBPN_FB_SDR_VEP = VEP_DIR / f"chromBPNet_VEP_out/{VCF_NAME}_Fibroblasts.mean.variant_scores.tsv"

#finetuned AG (workflows/alphagenome_ft/04_vep.sh)
AG_FT_SDR_VEP = VEP_DIR / f"AG_ft_VEP_out/VEP_AG_ft_fold_1_{VCF_NAME}.csv"

#Borzoi (workflows/borzoi/04_vep.sh)
BORZOI_SDR_VEP = VEP_DIR / f"Borzoi_VEP/VEP_Borzoi_{VCF_NAME}.csv"

#gtf for the strand matching of the fine-tuned AG RNA tracks
gtf = get_gtf(paths.gtf, chrom_filter="autosomesXY", gene_type_filter=None, feature=None)
gtf.loc[gtf.gene_name == "FAM69B", "gene_name"] =  "DIPK1B" #one of the sdr genes that has a differnt alias in the gtf

In [ ]:
obs = pd.read_csv(SDR_VCF_META_FILE)
obs_CM = obs[obs.cell_type == "CM"]
obs_FB = obs[obs.cell_type == "FB"]

In [ ]:
#pretrined AG prep
pred_ag = pd.read_csv(AG_pre_HEART_SDR_VEP).rename(columns={"gene_name": "Gene"})
pred_CM_ag = pd.read_csv(AG_pre_CM_SDR_VEP).rename(columns={"gene_name": "Gene"})
pred_FB_ag = pd.read_csv(AG_pre_FB_SDR_VEP).rename(columns={"gene_name": "Gene"})

pred_ag = pivot_tf_scores(pred_ag)
pred_CM_ag = pivot_tf_scores(pred_CM_ag)
pred_FB_ag = pivot_tf_scores(pred_FB_ag)

#add all other modalites to sdr data suing naive predcition of all heart tracks
naive_merge = pd.merge(pred_ag, obs, how ="inner", on = ["Gene", "CHROM", "POS", "REF", "ALT"])
naive_merge["pred_sign"] = np.sign(naive_merge.RNA_raw_score)
naive_merge["obs_sign"] = np.sign(naive_merge["logFC"])

naive_merge = add_abs_score_columns(naive_merge).reset_index(drop=True)

#add all other modalites to sdr data suing ct specific predcitions for CM and FB
CM_rna = pd.merge(obs_CM, pred_CM_ag, how ="inner", on = ["Gene", "CHROM", "POS", "REF", "ALT"])
FB_rna = pd.merge(obs_FB, pred_FB_ag, how ="inner", on = ["Gene", "CHROM", "POS", "REF", "ALT"])

ct_merge = pd.concat([CM_rna, FB_rna])

ct_merge["pred_sign"] = np.sign(ct_merge.RNA_raw_score)
ct_merge["obs_sign"] = np.sign(ct_merge["logFC"])

ct_merge = add_abs_score_columns(ct_merge).reset_index(drop=True)

#prep ag pr naive for causality
ag_pre_naive_pred = pred_ag.drop(["gene_id", "variant_name", *[col for col in pred_ag.columns if "quantile" in col]],axis = 1)
ag_pre_naive_pred.rename(columns={col: f"{col}_naive" for col in ag_pre_naive_pred.columns if "raw_score" in col}, inplace=True) 
pred_FB_ag["cell_type"] = "FB"
pred_CM_ag["cell_type"] = "CM"

ag_pre_ct_pred = pd.concat([pred_CM_ag, pred_FB_ag])
ag_pre_ct_pred.drop(["gene_id", "variant_name", *[col for col in ag_pre_ct_pred.columns if "quantile" in col]],axis = 1, inplace=True)
ag_pre_ct_pred.rename(columns={col: f"{col}_ct" for col in ag_pre_ct_pred.columns if "raw_score" in col}, inplace=True)

In [ ]:
#chromBPNet prep
#load chrombpnets
pred_cm_cbpn = pd.read_csv(cBPN_CM_SDR_VEP, sep = "\t")
pred_cm_cbpn = format_chrombpnet_vep(pred_cm_cbpn, keep_abs=True)

pred_fb_cbpn = pd.read_csv(cBPN_FB_SDR_VEP, sep = "\t")
pred_fb_cbpn = format_chrombpnet_vep(pred_fb_cbpn, keep_abs=True)

pred_cm_cbpn["cell_type"] = "CM"
pred_fb_cbpn["cell_type"] = "FB"

pred_cbpn = pd.concat([pred_cm_cbpn, pred_fb_cbpn])

#drop pahsing
pred_cbpn["variant_id"] = strip_phasing_suffix(pred_cbpn["variant_id"])
#agg pahsing
pred_cbpn = pred_cbpn.groupby(["variant_id", "cell_type", "CHROM", "POS", "REF", "ALT"], observed=True, dropna=False).agg({
    "COUNTS_cPBN_raw_score": "mean",
    "JSD_cPBN_raw_score": "mean",
    "COUNTSxJSD_cPBN_raw_score": "mean",
}).reset_index()

cbpn_merge = pd.merge(obs,
         pred_cbpn,
         how ="inner",
         on = ["variant_id", "cell_type","CHROM", "POS", "REF", "ALT"]).reset_index(drop=True)

In [ ]:
#fintuned alpahagenome prep: CM/FB tracks, Cut&Tag days averaged, strand matched RNA genes
ag_ft_full = format_alphagenome_ft_vep_per_gene(pd.read_csv(AG_FT_SDR_VEP), gtf)

#merge with obs data
agft_merge = pd.merge(ag_ft_full, obs, how ="inner", on = ["Gene", "variant_id", "cell_type"])#.drop_duplicates()

In [ ]:
#Borzoi prep: CM/FB tracks, Cut&Tag days averaged, one row per variant, cell type and gene
borzoi_pred = format_borzoi_vep_per_gene(pd.read_csv(BORZOI_SDR_VEP))

#merge with obs
borzoi_merge_rna = pd.merge(borzoi_pred, obs, how ="inner", on = ["Gene", "variant_id", "cell_type"])

# SDR effect size predcition

## AG pretrained

In [ ]:
#celltype matched ag pretriend
df = ct_merge
strat_dict = {
    "All Vars": df.index,
    "TSS Distance < 35kb": (df["dist_target_gene_TSS"].abs() <= 35_000),
    "TSS Distance 35kb-100kb": (df["dist_target_gene_TSS"].abs() > 35_000) & (df["dist_target_gene_TSS"].abs() <= 100_000),
    "TSS Distance 100kb-175kb": (df["dist_target_gene_TSS"].abs() > 100_000) & (df["dist_target_gene_TSS"].abs() <= 175_000),
    "TSS Distance 175kb-350kb": (df["dist_target_gene_TSS"].abs() > 175_000) & (df["dist_target_gene_TSS"].abs() <= 350_000),
    "TSS Distance > 350kb": (df["dist_target_gene_TSS"].abs() > 350_000),
}

# 2. Call the function
plot_scatter_panel(
    df=df, 
    x='RNA_raw_score', 
    y='logFC', 
    hue='var_type', 
    stratification_dict=strat_dict,
    cols=6 # Adjust this based on how wide you want the grid
)

In [ ]:
#naive all heart ag pretreinad
df = naive_merge
strat_dict = {
    "All Vars": df.index,
    "TSS Distance < 35kb": (df["dist_target_gene_TSS"].abs() <= 35_000),
    "TSS Distance 35kb-100kb": (df["dist_target_gene_TSS"].abs() > 35_000) & (df["dist_target_gene_TSS"].abs() <= 100_000),
    "TSS Distance 100kb-175kb": (df["dist_target_gene_TSS"].abs() > 100_000) & (df["dist_target_gene_TSS"].abs() <= 175_000),
    "TSS Distance 175kb-350kb": (df["dist_target_gene_TSS"].abs() > 175_000) & (df["dist_target_gene_TSS"].abs() <= 350_000),
    "TSS Distance > 350kb": (df["dist_target_gene_TSS"].abs() > 350_000),
}

# 2. Call the function
plot_scatter_panel(
    df=df, 
    x='RNA_raw_score', 
    y='logFC', 
    hue='var_type', 
    stratification_dict=strat_dict,
    cols=6 # Adjust this based on how wide you want the grid
)

In [ ]:
df = ct_merge
df = df.reset_index(drop=True)
strat_dict = {
    "All Variants": df.index,
    "cCRE Promoter": (df["cCRE_class"] == "Promoter") | (df["cCRE_class"] =="CA-H3K4me3") ,
    "cCRE Enhancer": (df["cCRE_class"] == "Distal enhancer") | (df["cCRE_class"] == "Proximal enhancer"),
    "Cardioid ATAC eak": df["is_cardioid_CA"].fillna(False),
    "Cardioid Enhancer": df["in_cardioid_enhancer"].fillna(False),
    "Cardioid H3K27ac +": df["is_cardioid_H3K27ac"].fillna(False),
}

# 2. Call the function
plot_scatter_panel(
    df=df, 
    x='RNA_raw_score', 
    y='logFC', 
    hue='var_type', 
    stratification_dict=strat_dict,
    save_fig=False,
    #fig_path=FIGURE_PATH + "AG_pretrained_RNAscore_SDR007010_Effect_size_scatter.pdf",
    cols=6
    
)

In [ ]:
df = naive_merge
df = df.reset_index(drop=True)
strat_dict = {
    "All Variants": df.index,
    "cCRE Promoter": (df["cCRE_class"] == "Promoter") | (df["cCRE_class"] =="CA-H3K4me3") ,
    "cCRE Enhancer": (df["cCRE_class"] == "Distal enhancer") | (df["cCRE_class"] == "Proximal enhancer"),
    "Cardioid ATAC eak": df["is_cardioid_CA"].fillna(False),
    "Cardioid Enhancer": df["in_cardioid_enhancer"].fillna(False),
    "Cardioid H3K27ac +": df["is_cardioid_H3K27ac"].fillna(False),
}

# 2. Call the function
plot_scatter_panel(
    df=df, 
    x='RNA_raw_score', 
    y='logFC', 
    hue='var_type', 
    stratification_dict=strat_dict,
    save_fig=False,
    #fig_path=FIGURE_PATH + "AG_pretrained_RNAscore_SDR007010_Effect_size_scatter.pdf",
    cols=6
    
)

In [ ]:
df = naive_merge
df = df[df.var_type == "positive"]
df = df.reset_index(drop=True)
strat_dict = create_tss_stratifications(df, n = 4)

# 2. Call the function
plot_scatter_panel(
    df=df, 
    x='RNA_raw_score', 
    y='logFC', 
    hue='var_type', 
    stratification_dict=strat_dict,
    save_fig=False,
    #fig_path=FIGURE_PATH + "AG_pretrained_RNAscore_SDR007010_Effect_size_scatter.pdf",
    cols=6
    
)

## AG finetuned  

In [ ]:
df = agft_merge
strat_dict = {
    "All Vars": df.index,
    "TSS Distance < 35kb": (df["dist_target_gene_TSS"].abs() <= 35_000),
    "TSS Distance 35kb-100kb": (df["dist_target_gene_TSS"].abs() > 35_000) & (df["dist_target_gene_TSS"].abs() <= 100_000),
    "TSS Distance 100kb-175kb": (df["dist_target_gene_TSS"].abs() > 100_000) & (df["dist_target_gene_TSS"].abs() <= 175_000),
    "TSS Distance 175kb-350kb": (df["dist_target_gene_TSS"].abs() > 175_000) & (df["dist_target_gene_TSS"].abs() <= 350_000),
    "TSS Distance > 350kb": (df["dist_target_gene_TSS"].abs() > 350_000),
}

# 2. Call the function
plot_scatter_panel(
    df=df, 
    x='AG_ft_RNA_score', 
    y='logFC', 
    hue='var_type', 
    stratification_dict=strat_dict,
    cols=6 # Adjust this based on how wide you want the grid
)

In [ ]:
df = agft_merge
strat_dict = {
    "All Variants": df.index,
    "cCRE Promoter": (df["cCRE_class"] == "Promoter") | (df["cCRE_class"] =="CA-H3K4me3") ,
    "cCRE Enhancer": (df["cCRE_class"] == "Distal enhancer") | (df["cCRE_class"] == "Proximal enhancer"),
    "Cardioid ATAC eak": df["is_cardioid_CA"].fillna(False),
    "Cardioid Enhancer": df["in_cardioid_enhancer"].fillna(False),
    "Cardioid H3K27ac +": df["is_cardioid_H3K27ac"].fillna(False),
}

plot_scatter_panel(
    df=df, 
    x='AG_ft_RNA_score', 
    y='logFC', 
    hue='var_type', 
    stratification_dict=strat_dict,
    cols=6 # Adjust this based on how wide you want the grid
)

## Borzoi finetuned

In [ ]:
df = borzoi_merge_rna
strat_dict = {
    "All Vars": df.index,
    "TSS Distance < 35kb": (df["dist_target_gene_TSS"].abs() <= 35_000),
    "TSS Distance 35kb-100kb": (df["dist_target_gene_TSS"].abs() > 35_000) & (df["dist_target_gene_TSS"].abs() <= 100_000),
    "TSS Distance 100kb-175kb": (df["dist_target_gene_TSS"].abs() > 100_000) & (df["dist_target_gene_TSS"].abs() <= 175_000),
    "TSS Distance 175kb-350kb": (df["dist_target_gene_TSS"].abs() > 175_000) & (df["dist_target_gene_TSS"].abs() <= 350_000),
    "TSS Distance > 350kb": (df["dist_target_gene_TSS"].abs() > 350_000),
}

plot_scatter_panel(
    df=df, 
    x='logSED_score', 
    y='logFC', 
    hue='var_type', 
    stratification_dict=strat_dict,
    cols=6 # Adjust this based on how wide you want the grid
)

In [ ]:
df = borzoi_merge_rna
strat_dict = {
    "All Variants": df.index,
    "cCRE Promoter": (df["cCRE_class"] == "Promoter") | (df["cCRE_class"] =="CA-H3K4me3") ,
    "cCRE Enhancer": (df["cCRE_class"] == "Distal enhancer") | (df["cCRE_class"] == "Proximal enhancer"),
    "Cardioid ATAC eak": df["is_cardioid_CA"].fillna(False),
    "Cardioid Enhancer": df["in_cardioid_enhancer"].fillna(False),
    "Cardioid H3K27ac +": df["is_cardioid_H3K27ac"].fillna(False),
}

plot_scatter_panel(
    df=df, 
    x='logSED_score', 
    y='logFC', 
    hue='var_type', 
    stratification_dict=strat_dict,
    cols=6
)

In [ ]:
df = borzoi_merge_rna
df["abs_logFC"] = abs(df["logFC"])
strat_dict = {
    "All Vars": df.index,
    "TSS Distance < 35kb": (df["dist_target_gene_TSS"].abs() <= 35_000),
    "TSS Distance 35kb-100kb": (df["dist_target_gene_TSS"].abs() > 35_000) & (df["dist_target_gene_TSS"].abs() <= 100_000),
    "TSS Distance 100kb-175kb": (df["dist_target_gene_TSS"].abs() > 100_000) & (df["dist_target_gene_TSS"].abs() <= 175_000),
    "TSS Distance 175kb-350kb": (df["dist_target_gene_TSS"].abs() > 175_000) & (df["dist_target_gene_TSS"].abs() <= 350_000),
    "TSS Distance > 350kb": (df["dist_target_gene_TSS"].abs() > 350_000),
    "All Variants": df.index,
    "cCRE Promoter": (df["cCRE_class"] == "Promoter") | (df["cCRE_class"] =="CA-H3K4me3") ,
    "cCRE Enhancer": (df["cCRE_class"] == "Distal enhancer") | (df["cCRE_class"] == "Proximal enhancer"),
    "Cardioid ATAC eak": df["is_cardioid_CA"].fillna(False),
    "Cardioid Enhancer": df["in_cardioid_enhancer"].fillna(False),
    "Cardioid H3K27ac +": df["is_cardioid_H3K27ac"].fillna(False),
}
plot_scatter_panel(
    df=df, 
    x='L2ED_score', 
    y='abs_logFC', 
    hue='var_type', 
    stratification_dict=strat_dict,
    cols=6 # Adjust this based on how wide you want the grid
)

# Effect sign prediction

In [ ]:
#add sdr log FC sign
naive_merge["obs_sign"] = np.sign(naive_merge["logFC"])
ct_merge["obs_sign"] = np.sign(ct_merge["logFC"])
borzoi_merge_rna["obs_sign"] = np.sign(borzoi_merge_rna["logFC"])
agft_merge["obs_sign"] = np.sign(agft_merge["logFC"])

In [ ]:
#merged RNA rediction of AG_pre AF_ft and Borwoi
ag_rna_merge = pd.merge(ct_merge[[col for col in ct_merge.columns if ("score" not in col or "RNA" in col or "CADD" in col)]],
         naive_merge[[col for col in naive_merge.columns if ("score" not in col or "RNA" in col or "CADD" in col)]], 
         how = "inner", on = ["obs_sign", *[col for col in naive_merge.columns if (("sign" not in col and "score" not in col) or "CADD" in col)]],
         suffixes=("_ct", "_naive")
        )
ag_rna_merge.drop([col for col in ag_rna_merge.columns if "quantile" in col], axis=1, inplace=True)
rna_shared_cols = set(ag_rna_merge.columns) & set(borzoi_merge_rna.columns) & set(agft_merge.columns)
rna_causality = pd.merge(ag_rna_merge, borzoi_merge_rna, how = "inner", on = list(rna_shared_cols))
rna_causality = pd.merge(rna_causality, agft_merge[[col for col in agft_merge.columns if ("score" not in col or "RNA" in col or "CADD" in col)]],
                         how = "inner", on = list(rna_shared_cols))
rna_causality["has_effect"] = rna_causality["var_type"] == "positive"
rna_causality["positive_sign"] = np.sign(rna_causality["logFC"]) == 1

## AG pretrained

In [ ]:
df = naive_merge
df = df[df.var_type == "positive"]

score = 'RNA_raw_score'
strat_dict = {
    "All Vars": df.index,
    # "TSS Distance < 90kb": (df["dist_target_gene_TSS"].abs() <= 90_000),
    # "TSS Distance 90kb-200kb": (df["dist_target_gene_TSS"].abs() > 90_000) & (df["dist_target_gene_TSS"].abs() <= 200_000),
    # "TSS Distance > 200kb": (df["dist_target_gene_TSS"].abs() > 200_000),
    "All Vars (> 5% absmax log2FC)": (df["logFC"].abs() >= df["logFC"].abs().max() * 0.05),
    # "TSS Distance < 90kb (> 5%)": (df["logFC"].abs() >= df["logFC"].abs().max() * 0.05),
    # "TSS Distance 90kb-200kb (> 5%)": (df["dist_target_gene_TSS"].abs() > 90_000) & (df["dist_target_gene_TSS"].abs() <= 200_000) & (df["logFC"].abs() >= df["logFC"].abs().max() * 0.05),
    # "TSS Distance > 200kb (> 5%)": (df["dist_target_gene_TSS"].abs() > 200_000) & (df["logFC"].abs() >= df["logFC"].abs().max() * 0.05),
}
plot_scatter_panel(
    df=df, 
    x='obs_sign', 
    y=score, 
    hue = "logFC",
    stratification_dict=strat_dict,
    cols=6 # Adjust this based on how wide you want the grid
)

In [ ]:
df = ct_merge
df = df[df.var_type == "positive"]

score = 'RNA_raw_score'
strat_dict = {
    "All Vars": df.index,
    # "TSS Distance < 90kb": (df["dist_target_gene_TSS"].abs() <= 90_000),
    # "TSS Distance 90kb-200kb": (df["dist_target_gene_TSS"].abs() > 90_000) & (df["dist_target_gene_TSS"].abs() <= 200_000),
    # "TSS Distance > 200kb": (df["dist_target_gene_TSS"].abs() > 200_000),
    "All Vars (> 5% absmax log2FC)": (df["logFC"].abs() >= df["logFC"].abs().max() * 0.05),
    # "TSS Distance < 90kb (> 5%)": (df["logFC"].abs() >= df["logFC"].abs().max() * 0.05),
    # "TSS Distance 90kb-200kb (> 5%)": (df["dist_target_gene_TSS"].abs() > 90_000) & (df["dist_target_gene_TSS"].abs() <= 200_000) & (df["logFC"].abs() >= df["logFC"].abs().max() * 0.05),
    # "TSS Distance > 200kb (> 5%)": (df["dist_target_gene_TSS"].abs() > 200_000) & (df["logFC"].abs() >= df["logFC"].abs().max() * 0.05),
}
plot_scatter_panel(
    df=df, 
    x='obs_sign', 
    y=score, 
    hue = "logFC",
    stratification_dict=strat_dict,
    cols=6
)

## AG finetuned

In [ ]:
df = agft_merge
df = df[df.var_type == "positive"]

score = 'AG_ft_RNA_score'
strat_dict = {
    "All Vars": df.index,
    # "TSS Distance < 90kb": (df["dist_target_gene_TSS"].abs() <= 90_000),
    # "TSS Distance 90kb-200kb": (df["dist_target_gene_TSS"].abs() > 90_000) & (df["dist_target_gene_TSS"].abs() <= 200_000),
    # "TSS Distance > 200kb": (df["dist_target_gene_TSS"].abs() > 200_000),
    "All Vars (> 5% absmax log2FC)": (df["logFC"].abs() >= df["logFC"].abs().max() * 0.05),
    # "TSS Distance < 90kb (> 5%)": (df["logFC"].abs() >= df["logFC"].abs().max() * 0.05),
    # "TSS Distance 90kb-200kb (> 5%)": (df["dist_target_gene_TSS"].abs() > 90_000) & (df["dist_target_gene_TSS"].abs() <= 200_000) & (df["logFC"].abs() >= df["logFC"].abs().max() * 0.05),
    # "TSS Distance > 200kb (> 5%)": (df["dist_target_gene_TSS"].abs() > 200_000) & (df["logFC"].abs() >= df["logFC"].abs().max() * 0.05),
}
plot_scatter_panel(
    df=df, 
    x='obs_sign', 
    y=score, 
    hue = "logFC",
    stratification_dict=strat_dict,
    cols=6
)

## Borzoi

In [ ]:
df = borzoi_merge_rna
df = df[df.var_type == "positive"]

score = 'logSED_score'
strat_dict = {
    "All Vars": df.index,
    # "TSS Distance < 90kb": (df["dist_target_gene_TSS"].abs() <= 90_000),
    # "TSS Distance 90kb-200kb": (df["dist_target_gene_TSS"].abs() > 90_000) & (df["dist_target_gene_TSS"].abs() <= 200_000),
    # "TSS Distance > 200kb": (df["dist_target_gene_TSS"].abs() > 200_000),
    "All Vars (> 5% absmax log2FC)": (df["logFC"].abs() >= df["logFC"].abs().max() * 0.05),
    # "TSS Distance < 90kb (> 5%)": (df["logFC"].abs() >= df["logFC"].abs().max() * 0.05),
    # "TSS Distance 90kb-200kb (> 5%)": (df["dist_target_gene_TSS"].abs() > 90_000) & (df["dist_target_gene_TSS"].abs() <= 200_000) & (df["logFC"].abs() >= df["logFC"].abs().max() * 0.05),
    # "TSS Distance > 200kb (> 5%)": (df["dist_target_gene_TSS"].abs() > 200_000) & (df["logFC"].abs() >= df["logFC"].abs().max() * 0.05),
}
plot_scatter_panel(
    df=df, 
    x='obs_sign', 
    y=score, 
    hue = "logFC",
    stratification_dict=strat_dict,
    cols=6
)

### Comparisons

In [ ]:
plot_AUROCs(rna_causality[rna_causality.var_type == "positive"],
            true_label="positive_sign",
            score_columns=['RNA_raw_score_ct', 'RNA_raw_score_naive',  'AG_ft_RNA_score', "logSED_score"])

In [ ]:
plot_AUPRCs(rna_causality[rna_causality.var_type == "positive"],
            true_label="positive_sign",
            score_columns=['RNA_raw_score_ct', 'RNA_raw_score_naive',  'AG_ft_RNA_score', "logSED_score"])

In [ ]:
plot_AUROCs(rna_causality[(rna_causality.var_type == "positive") & (rna_causality["logFC"].abs() >= rna_causality["logFC"].abs().max() * 0.05)],
            true_label="positive_sign",
            score_columns=['RNA_raw_score_ct', 'RNA_raw_score_naive',  'AG_ft_RNA_score', "logSED_score"])

In [ ]:
plot_AUPRCs(rna_causality[(rna_causality.var_type == "positive") & (rna_causality["logFC"].abs() >= rna_causality["logFC"].abs().max() * 0.05)],
            true_label="positive_sign",
            score_columns=['RNA_raw_score_ct', 'RNA_raw_score_naive',  'AG_ft_RNA_score', "logSED_score"])

In [ ]:
#compare calssification performance based on auprc
df = rna_causality[rna_causality.var_type == "positive"]

# Call the function
matrix_df = plot_stratified_auprc_heatmap(
    df=df, 
    true_label="positive_sign", 
    score_columns=['RNA_raw_score_ct', 'RNA_raw_score_naive',  'AG_ft_RNA_score', "logSED_score"], 
    strat_dict={
        "All Vars": df.index,
        "All Vars\n(> 5% absmax log2FC)": (df["logFC"].abs() >= df["logFC"].abs().max() * 0.05),
    },
    alternate_score_names = ["AG_pre RNA cell type matched", "AG_pre RNA all heart tracks", "AG_ft RNA cell type matched", "Borzoi logSED"],
    title="SDR-seq RNA effect sign predcition",
    save_fig=False,
    #fig_path=FIGURE_PATH + 'AG_pretrained_vs_Borzoi_ft_SDR0070010_RNA_Causality_AUPRC_heatmap.pdf'
)

In [ ]:
#calssification cutoff at score = 0
fig, axes, results = plot_contingency_grid(
    rna_causality[rna_causality.var_type == "positive"],
    true_label="positive_sign",
    score_cols=['RNA_raw_score_ct', 'RNA_raw_score_naive',  'AG_ft_RNA_score', "logSED_score"],
    threshold=0.0,                       
    titles={'RNA_raw_score_ct': "AG_pre RNA cell type matched",
            'RNA_raw_score_naive': "AG_pre RNA all heart tracks",
            'AG_ft_RNA_score':"AG_ft RNA cell type matched",
            "logSED_score": "Borzoi logSED"},
    ncols=4,figsize_per_cell=2.5, fontsize=11
)

# plt.savefig("/mnt/user-data/outputs/demo_contingency_grid.png", dpi=150)

In [ ]:
# f1 based optimal calssification cutoff
fig, axes, results = plot_contingency_grid(
    rna_causality[rna_causality.var_type == "positive"],
    true_label="positive_sign",
    score_cols=['RNA_raw_score_ct', 'RNA_raw_score_naive',  'AG_ft_RNA_score', "logSED_score", "L2ED_score"],
    optimal_threshold=True,  # optimal for all
    titles={'RNA_raw_score_ct': "AG_pre RNA cell type matched",
            'RNA_raw_score_naive': "AG_pre RNA all heart tracks",
            'AG_ft_RNA_score':"AG_ft RNA cell type matched",
            "logSED_score": "Borzoi logSED",
            "L2ED_score": "Borzoi L2ED"},
    ncols=5,figsize_per_cell=2.5, fontsize=11
)

# plt.savefig("/mnt/user-data/outputs/demo_contingency_grid.png", dpi=150)

In [ ]:
for col, r in results.items():
    print(col, "cutoff:", r["threshold"], "p-value:", r["chi2_pvalue"])

# Causality prediction

In [ ]:
combined_pred = pd.merge(ag_ft_full.drop(["strand"], axis = 1), borzoi_pred, how = "outer", on = ["variant_id", "Gene", "cell_type"]) #merge ag ft and borzoi
combined_pred = pd.merge(combined_pred, pred_cbpn, how = "outer", on = ["variant_id", "cell_type"]) #add chombnet
combined_pred = pd.merge(combined_pred, ag_pre_naive_pred, how = "outer", on = ["Gene", "CHROM", "POS", "REF", "ALT"]) #add ag pre naicve (mathc by chrom pos etc becuase of differ ent vairnt id
combined_pred = pd.merge(combined_pred, ag_pre_ct_pred.dropna(axis=1), how = "outer", on = ["Gene", "CHROM", "POS", "REF", "ALT", "cell_type"]) #add ag pre ct (mathc by chrom pos etc becuase of differ ent vairnt id

In [ ]:
causality = pd.merge(combined_pred, obs, how = "right", on = ["variant_id", "cell_type", "Gene", "CHROM", "POS", "REF", "ALT"])
#pick the abs max rna prediction per variant to collapse genes in RNA
idx = causality.groupby(["variant_id", "cell_type"])["logFC"].apply(lambda x: x.abs().idxmax())
causality = causality.loc[idx].reset_index(drop= True)

In [ ]:
#add absolute columns
all_score_cols = list(causality.columns[(causality.columns.str.contains("_score")) & (causality.columns != "CADD_PHRED_score")])
abs_cols = [f"abs_{col}" for col in all_score_cols]
causality[abs_cols] = causality[all_score_cols].abs()

In [ ]:
#add has_effect as bool column for all positive vars
causality["has_effect"] = causality["var_type"] == "positive"

In [ ]:
causality["1/TSS"] = 1 / causality["dist_target_gene_TSS"].abs()

## AG pretrained ct matched causality

In [ ]:
ag_ct_cols = [col for col in abs_cols if "ct" in col and "abs" in col]
ag_pre_causality = causality.dropna(subset=ag_ct_cols)

In [ ]:
plot_AUPRCs(ag_pre_causality, true_label="has_effect", score_columns=['abs_RNA_raw_score_ct','abs_ATAC_raw_score_ct', 'abs_POLR2A_TF_raw_score_ct'], n_cols=3)

In [ ]:
box_grid(ag_pre_causality, x_col="has_effect", y_cols=[ 'abs_POLR2A_TF_raw_score_ct','abs_ATAC_raw_score_ct','abs_RNA_raw_score_ct'], n_cols = 3, alternative ="greater")

In [ ]:
power_results, required_n = mannwhitney_power_grid(
    ag_pre_causality,
    x_col="has_effect",
    y_cols=['abs_POLR2A_TF_raw_score_ct','abs_ATAC_raw_score_ct','abs_RNA_raw_score_ct'],
    n_cols=4,
    alternative="greater",
    n_range=(10, 20, 50, 100, 150, 200, 300, 400, 500),
    n_sims=1000,
    alpha=0.05,
    target_power=0.8,
)

In [ ]:
df = ag_pre_causality

strat_dict = {
"All Vars": df.index,
"TSS Distance < 100kb": (df["dist_target_gene_TSS"].abs() <= 100_000),
"TSS Distance 100kb-200kb": (df["dist_target_gene_TSS"].abs() > 100_000) & (df["dist_target_gene_TSS"].abs() <= 200_000),
"TSS Distance > 200kb": (df["dist_target_gene_TSS"].abs() > 200_000),
}

matrix_df = plot_stratified_auprc_heatmap(
    df=df, 
    true_label="has_effect", 
    score_columns=ag_ct_cols, 
    strat_dict=strat_dict, 
    title="AlphaGenome pretrained ct causality",
    save_fig=False,
    #fig_path=FIGURE_PATH + 'AG_pretrained_vs_Borzoi_ft_SDR0070010_RNA_Causality_AUPRC_heatmap.pdf'
)

In [ ]:
fig, axes, results = plot_contingency_grid(
    ag_pre_causality,#[ag_ft_causality.dist_target_gene_TSS < 100_000],
    true_label="has_effect",
    score_cols=['abs_POLR2A_TF_raw_score_ct','abs_ATAC_raw_score_ct','abs_RNA_raw_score_ct'],
    optimal_threshold=True,  # optimal for all
    # titles={'RNA_raw_score_ct': "AG_pre RNA cell type matched",
    #         'RNA_raw_score_naive': "AG_pre RNA all heart tracks",
    #         'AG_ft_RNA_score':"AG_ft RNA cell type matched",
    #         "logSED_score": "Borzoi logSED",
    #         "L2ED_score": "Borzoi L2ED"},
    ncols=1,figsize_per_cell=3, fontsize=11
)

In [ ]:
ag_ct_naive_cols = ['abs_H3K36me3_HIST_raw_score_naive','abs_H3K36me3_HIST_raw_score_ct',
 'abs_RNA_raw_score_naive','abs_RNA_raw_score_ct',
 'abs_ATAC_raw_score_naive','abs_ATAC_raw_score_ct',
 'abs_POLR2A_TF_raw_score_naive','abs_POLR2A_TF_raw_score_ct']

ag_pre_causality_naive = causality.dropna(subset=ag_ct_naive_cols)

df = ag_pre_causality_naive
matrix_df = plot_stratified_auprc_heatmap(
    df=df, 
    true_label="has_effect", 
    score_columns=ag_ct_naive_cols, 
    strat_dict={
        "Cardimyocytes": (df["cell_type"] == "CM"),
        "Fibroblasts": (df["cell_type"] == "FB"),

    },
    title="AlphaGenome pretrained ct vs naive causality",
    save_fig=False,
    #fig_path=FIGURE_PATH + 'AG_pretrained_vs_Borzoi_ft_SDR0070010_RNA_Causality_AUPRC_heatmap.pdf'
)

## AG ft causality

In [ ]:
ag_ft_cols = [col for col in abs_cols if "AG_ft" in col and "abs" in col]
ag_ft_causality = causality.dropna(subset=ag_ft_cols)

In [ ]:
plot_AUPRCs(ag_ft_causality, true_label="has_effect", score_columns=ag_ft_cols, n_cols=4)

In [ ]:
box_grid(ag_ft_causality, x_col="has_effect", y_cols=['abs_AG_ft_ATAC_score','abs_AG_ft_RNA_score','abs_AG_ft_H3K27ac_score', 'abs_AG_ft_H3K4me3_score'], n_cols = 4, alternative ="greater")

In [ ]:
power_results, required_n = mannwhitney_power_grid(
    ag_ft_causality,
    x_col="has_effect",
    y_cols=['abs_AG_ft_ATAC_score','abs_AG_ft_RNA_score',
            'abs_AG_ft_H3K27ac_score', 'abs_AG_ft_H3K4me3_score'],
    n_cols=4,
    alternative="greater",
    n_range=(100, 200, 300, 400, 500, 600, 700, 800, 1000),
    n_sims=1000,
    alpha=0.05,
    target_power=0.8,
)

In [ ]:
ag_ft_causality.has_effect.value_counts()

In [ ]:
df = ag_ft_causality
strat_dict = {
"All Vars": df.index,
"TSS Distance < 100kb": (df["dist_target_gene_TSS"].abs() <= 100_000),
"TSS Distance 100kb-200kb": (df["dist_target_gene_TSS"].abs() > 100_000) & (df["dist_target_gene_TSS"].abs() <= 200_000),
"TSS Distance > 200kb": (df["dist_target_gene_TSS"].abs() > 200_000),
}
matrix_df = plot_stratified_auprc_heatmap(
    df=df, 
    true_label="has_effect", 
    score_columns=ag_ft_cols, 
    strat_dict=strat_dict, 
    title="AlphaGenome finetuned causality",
    save_fig=False,
    #fig_path=FIGURE_PATH + 'AG_pretrained_vs_Borzoi_ft_SDR0070010_RNA_Causality_AUPRC_heatmap.pdf'
)

In [ ]:
fig, axes, results = plot_contingency_grid(
    ag_ft_causality,#[ag_ft_causality.dist_target_gene_TSS < 50_000],
    true_label="has_effect",
    score_cols=['abs_AG_ft_ATAC_score','abs_AG_ft_RNA_score','abs_AG_ft_H3K27ac_score', 'abs_AG_ft_H3K4me3_score'],
    optimal_threshold=True,  # optimal for all
    # titles={'RNA_raw_score_ct': "AG_pre RNA cell type matched",
    #         'RNA_raw_score_naive': "AG_pre RNA all heart tracks",
    #         'AG_ft_RNA_score':"AG_ft RNA cell type matched",
    #         "logSED_score": "Borzoi logSED",
    #         "L2ED_score": "Borzoi L2ED"},
    ncols=4,figsize_per_cell=2.5, fontsize=11
)

In [ ]:
#compare the alaphgenomes
ag_all_cols = ['abs_AG_ft_ATAC_score',  'abs_ATAC_raw_score_naive', 'abs_ATAC_raw_score_ct',
'abs_AG_ft_H3K27ac_score', 'abs_H3K27ac_HIST_raw_score_naive', 'abs_H3K27ac_HIST_raw_score_ct',
'abs_AG_ft_H3K4me3_score','abs_H3K4me3_HIST_raw_score_naive', 'abs_H3K4me3_HIST_raw_score_ct', 
'abs_AG_ft_RNA_score','abs_RNA_raw_score_naive', 'abs_RNA_raw_score_ct']

ag_all_causality = causality.dropna(subset=ag_all_cols)

df = ag_all_causality
matrix_df = plot_stratified_auprc_heatmap(
    df=df, 
    true_label="has_effect", 
    score_columns=ag_all_cols, 
    strat_dict={
        "Cardimyocytes": (df["cell_type"] == "CM"),
        "Fibroblasts": (df["cell_type"] == "FB"),
    },
    title="AlphaGenome pretrained vs finetuned causality",
    save_fig=False,
    #fig_path=FIGURE_PATH + 'AG_pretrained_vs_Borzoi_ft_SDR0070010_RNA_Causality_AUPRC_heatmap.pdf'
)

## Borzoi causality

In [ ]:
borzoi_cols = ['L2ED_score', 'abs_logSED_score', 'L2_score_RNA',  'abs_SUM_score_RNA', 'abs_logSUM_score_RNA',
 'L2_score_H3K27ac', 'abs_SUM_score_H3K27ac', 'abs_logSUM_score_H3K27ac',
 'L2_score_H3K4me3', 'abs_SUM_score_H3K4me3', 'abs_logSUM_score_H3K4me3',
 'L2_score_ATAC', 'abs_SUM_score_ATAC', 'abs_logSUM_score_ATAC',]
borzoi_causality = causality.dropna(subset=borzoi_cols)

In [ ]:
plot_AUPRCs(borzoi_causality, true_label="has_effect", score_columns=['abs_SUM_score_RNA', 'abs_logSED_score', 'abs_SUM_score_ATAC',
  'abs_SUM_score_H3K27ac', 
 'abs_SUM_score_H3K4me3', 
 ],
            n_cols=5)

In [ ]:
box_grid(borzoi_causality, x_col="has_effect", y_cols=['abs_SUM_score_RNA', 'abs_logSED_score', 'abs_SUM_score_ATAC',
  'abs_SUM_score_H3K27ac', 
 'abs_SUM_score_H3K4me3', 
 ], n_cols = 10, alternative ="greater")

In [ ]:
df = borzoi_causality

matrix_df = plot_stratified_auprc_heatmap(
    df=df, 
    true_label="has_effect", 
    score_columns=borzoi_cols, 
    strat_dict=create_tss_stratifications(df, n = 2), 
    title="Borzoi causality",
    save_fig=False,
    #fig_path=FIGURE_PATH + 'AG_pretrained_vs_Borzoi_ft_SDR0070010_RNA_Causality_AUPRC_heatmap.pdf'
)

In [ ]:
fig, axes, results = plot_contingency_grid(
    borzoi_causality,#[borzoi_causality.dist_target_gene_TSS < 60_000],
    true_label="has_effect",
    score_cols=['abs_SUM_score_RNA', 'abs_logSED_score', 'abs_SUM_score_ATAC',
  'abs_SUM_score_H3K27ac', 
 'abs_SUM_score_H3K4me3', 
 ],
    optimal_threshold=True,  # optimal for all
    # titles={'RNA_raw_score_ct': "AG_pre RNA cell type matched",
    #         'RNA_raw_score_naive': "AG_pre RNA all heart tracks",
    #         'AG_ft_RNA_score':"AG_ft RNA cell type matched",
    #         "logSED_score": "Borzoi logSED",
    #         "L2ED_score": "Borzoi L2ED"},
    ncols=5,figsize_per_cell=2.5, fontsize=11
)

## chromBPNets causality

In [ ]:
cbpn_cols = ['JSD_cPBN_raw_score','abs_COUNTS_cPBN_raw_score', 'abs_COUNTSxJSD_cPBN_raw_score']
cbpn_causality = causality.dropna(subset=cbpn_cols)

In [ ]:
plot_AUPRCs(cbpn_causality, true_label="has_effect", score_columns=cbpn_cols, n_cols=3)

In [ ]:
box_grid(cbpn_causality, x_col="has_effect", y_cols=cbpn_cols, n_cols = 3, alternative ="greater")

In [ ]:
power_results, required_n = mannwhitney_power_grid(
    cbpn_causality,
    x_col="has_effect",
    y_cols=cbpn_cols,
    n_cols=4,
    alternative="greater",
    n_range=(100, 200, 300, 400, 500, 600, 700, 800, 1000),
    n_sims=1000,
    alpha=0.05,
    target_power=0.8,
)

In [ ]:
df = cbpn_causality
matrix_df = plot_stratified_auprc_heatmap(
    df=df, 
    true_label="has_effect", 
    score_columns=cbpn_cols, 
    strat_dict = {
"All Vars": df.index,
"TSS Distance < 100kb": (df["dist_target_gene_TSS"].abs() <= 100_000),
"TSS Distance 100kb-200kb": (df["dist_target_gene_TSS"].abs() > 100_000) & (df["dist_target_gene_TSS"].abs() <= 200_000),
"TSS Distance > 200kb": (df["dist_target_gene_TSS"].abs() > 200_000),
}, 
    title="chromBPNet causality",
    alternate_score_names = ["JSD", "abs count logFC", "IES"],
    save_fig=False,
    #fig_path=FIGURE_PATH + 'AG_pretrained_vs_Borzoi_ft_SDR0070010_RNA_Causality_AUPRC_heatmap.pdf'
)

In [ ]:
fig, axes, results = plot_contingency_grid(
    cbpn_causality[cbpn_causality.dist_target_gene_TSS < 100_000],
    true_label="has_effect",
    score_cols=cbpn_cols,
    optimal_threshold=True,  # optimal for all
    # titles={'RNA_raw_score_ct': "AG_pre RNA cell type matched",
    #         'RNA_raw_score_naive': "AG_pre RNA all heart tracks",
    #         'AG_ft_RNA_score':"AG_ft RNA cell type matched",
    #         "logSED_score": "Borzoi logSED",
    #         "L2ED_score": "Borzoi L2ED"},
    ncols=3,figsize_per_cell=2.5, fontsize=11
)

## full model comparison causality

In [ ]:
full_score_col = [ 'abs_ATAC_raw_score_ct',
  'abs_AG_ft_ATAC_score',
 'abs_COUNTS_cPBN_raw_score',
 'abs_COUNTSxJSD_cPBN_raw_score',
 'JSD_cPBN_raw_score',
  'abs_RNA_raw_score_ct',
 'abs_AG_ft_RNA_score',
    'abs_H3K27ac_HIST_raw_score_ct',
  'abs_AG_ft_H3K27ac_score',
          'abs_H3K4me3_HIST_raw_score_ct',
  'abs_AG_ft_H3K4me3_score',
 '1/TSS']

names = [ 'AG pre',
  'AG ft',
 'chromBPNet (logFC)',
 'chromBPNet (IES)',
 'chromBPNet (JSD)',
  'AG pre',
  'AG ft',
  'AG pre',
  'AG ft',
  'AG pre',
  'AG ft',
 '1/TSS']

In [ ]:
full_comp_causality = causality.dropna(subset=full_score_col)

In [ ]:
df = full_comp_causality
matrix_df = plot_stratified_auprc_heatmap(
    df=df, 
    true_label="has_effect", 
    score_columns=full_score_col, 
    strat_dict= {
    "All Vars": df.index,
    "TSS Distance < 100kb": (df["dist_target_gene_TSS"].abs() <= 100_000),
    "TSS Distance 100kb-200kb": (df["dist_target_gene_TSS"].abs() > 100_000) & (df["dist_target_gene_TSS"].abs() <= 200_000),
    "TSS Distance > 200kb": (df["dist_target_gene_TSS"].abs() > 200_000),    "cCRE Enhancer": (df["cCRE_class"] == "Distal enhancer") | (df["cCRE_class"] == "Proximal enhancer"),
    "Cardioid ATAC Peaks": df["is_cardioid_CA"].fillna(False),
    "Cardioid ABC enhancer": df["in_cardioid_enhancer"].fillna(False),
    "Cardioid Proximal Enhancer": df["enhancer_regulatory_position"] == "proximal",
    "Cardioid Distal Enhancer": df["enhancer_regulatory_position"] == "distal",
  #  "Coding Region": (df["gtf_feature"] == "exon") | (df["gtf_feature"] =="start_codon") | (df["gtf_feature"] =="stop_codon") | (df["gtf_feature"] =="CDS"),
   # "Non-coding Region": ~((df["gtf_feature"] == "exon") | (df["gtf_feature"] =="start_codon") | (df["gtf_feature"] =="stop_codon") | (df["gtf_feature"] =="CDS")),
}, alternate_score_names=names,
    title=" causality",
    save_fig=False,
    #fig_path=FIGURE_PATH + 'AG_pretrained_vs_Borzoi_ft_SDR0070010_RNA_Causality_AUPRC_heatmap.pdf'
)

In [ ]:
fig, axes, results = plot_contingency_grid(
    full_comp_causality,
    true_label="has_effect",
    score_cols=full_score_col,
    optimal_threshold=True,  # optimal for all
    # titles={'RNA_raw_score_ct': "AG_pre RNA cell type matched",
    #         'RNA_raw_score_naive': "AG_pre RNA all heart tracks",
    #         'AG_ft_RNA_score':"AG_ft RNA cell type matched",
    #         "logSED_score": "Borzoi logSED",
    #         "L2ED_score": "Borzoi L2ED"},
    ncols=10,figsize_per_cell=2.5, fontsize=11
)

### RNA model causality comp using all var gene pairs

In [ ]:
#add absolute columns
rna_score_cols = list(rna_causality.columns[(rna_causality.columns.str.contains("_score")) & (rna_causality.columns.str.contains("RNA|L2ED|SED")) & ~(rna_causality.columns.str.contains("abs_"))])
abs_rna_cols = [f"abs_{col}" for col in rna_score_cols]
rna_causality[abs_rna_cols] = rna_causality[rna_score_cols].abs()

In [ ]:
rna_causality["1/TSS"] = 1 / rna_causality["dist_target_gene_TSS"].abs()

In [ ]:
#RNA
df = rna_causality[~rna_causality.variant_id.str.contains("chr10-97456968-T-C")].drop_duplicates()
# Call the function
matrix_df = plot_stratified_auprc_heatmap(
    df=df, 
    true_label="has_effect", 
    score_columns=["abs_AG_ft_RNA_score", "abs_RNA_raw_score_ct", "abs_logSED_score", "L2ED_score", "1/TSS"], 
     strat_dict={
        "All Variants": df.index,
"TSS Distance < 35kb": (df["dist_target_gene_TSS"].abs() <= 35_000),
"TSS Distance 35kb-50kb": (df["dist_target_gene_TSS"].abs() > 35_000) & (df["dist_target_gene_TSS"].abs() <= 50_000),
"TSS Distance > 50kb": (df["dist_target_gene_TSS"].abs() > 50_000),
        "cCRE Enhancer": (df["cCRE_class"] == "Distal enhancer") | (df["cCRE_class"] == "Proximal enhancer"),
        "Cardioid ATAC Peaks": df["is_cardioid_CA"].fillna(False),
        "Cardioid ABC enhancer": df["in_cardioid_enhancer"].fillna(False),
        "Cardioid Proximal Enhancer": df["enhancer_regulatory_position"] == "proximal",
        "Cardioid Distal Enhancer": df["enhancer_regulatory_position"] == "distal"
        }, 
    alternate_score_names = ["AG ft cell type matched", "AG pre cell type matched", "Borzoi abs logSED", "Borzoi L2ED", "1/TSS"],
    title="SDR-seq causality classifiation RNA tracks",
    save_fig=False,
    #fig_path=FIGURE_PATH + 'AG_pretrained_vs_Borzoi_ft_SDR0070010_RNA_Causality_AUPRC_heatmap.pdf'
)

In [ ]:
plot_AUPRCs(df, true_label="has_effect", score_columns=["abs_AG_ft_RNA_score", "abs_RNA_raw_score_ct", "abs_logSED_score", "L2ED_score", "1/TSS"], n_cols=10)

In [ ]:
plot_AUPRCs(df[df["enhancer_regulatory_position"] == "proximal"], true_label="has_effect", score_columns=["abs_AG_ft_RNA_score", "abs_RNA_raw_score_ct", "abs_logSED_score", "L2ED_score", "1/TSS"], n_cols=10)

In [ ]:
df.has_effect.value_counts()

In [ ]:
power_results, required_n = mannwhitney_power_grid(
    df,
    x_col="has_effect",
    y_cols=["abs_AG_ft_RNA_score", "abs_RNA_raw_score_ct", "abs_logSED_score", "L2ED_score",],
    n_cols=4,
    alternative="greater",
    n_range=(100, 200, 300, 400, 500, 600, 700, 800, 1000),
    n_sims=1000,
    alpha=0.05,
    target_power=0.8,
)

In [ ]:
fig, axes, results = plot_contingency_grid(
    df,
    true_label="has_effect",
    score_cols=["abs_AG_ft_RNA_score", "abs_logSED_score"],
    optimal_threshold=True,  # optimal for all
    # titles={'RNA_raw_score_ct': "AG_pre RNA cell type matched",
    #         'RNA_raw_score_naive': "AG_pre RNA all heart tracks",
    #         'AG_ft_RNA_score':"AG_ft RNA cell type matched",
    #         "logSED_score": "Borzoi logSED",
    #         "L2ED_score": "Borzoi L2ED"},
    ncols=1,figsize_per_cell=2.5, fontsize=11
)

In [ ]:
fig, axes, results = plot_contingency_grid(
    rna_causality[rna_causality["enhancer_regulatory_position"] == "proximal"],
    true_label="has_effect",
    score_cols=["abs_AG_ft_RNA_score", "abs_RNA_raw_score_ct", "abs_RNA_raw_score_naive", "abs_logSED_score", "L2ED_score", "1/TSS"],
    optimal_threshold=True,  # optimal for all
    # titles={'RNA_raw_score_ct': "AG_pre RNA cell type matched",
    #         'RNA_raw_score_naive': "AG_pre RNA all heart tracks",
    #         'AG_ft_RNA_score':"AG_ft RNA cell type matched",
    #         "logSED_score": "Borzoi logSED",
    #         "L2ED_score": "Borzoi L2ED"},
    ncols=10,figsize_per_cell=2.5, fontsize=11
)

In [ ]:
box_grid(rna_causality, x_col="has_effect", y_cols=["abs_AG_ft_RNA_score", "abs_RNA_raw_score_ct", "abs_RNA_raw_score_naive", "abs_logSED_score", "L2ED_score", "1/TSS"], n_cols = 10, alternative ="greater")

In [ ]:
box_grid(rna_causality[rna_causality["enhancer_regulatory_position"] == "proximal"], x_col="has_effect", y_cols=["abs_AG_ft_RNA_score", "abs_RNA_raw_score_ct", "abs_RNA_raw_score_naive", "abs_logSED_score", "L2ED_score", "1/TSS"], n_cols = 10, alternative ="greater")

# Check richter prioritized vars

We found in the only sigifacnt enriched CHD stratifaction on the richter vars a set of 233 unique DNVs. We wanna see whether they have an effect in SDR as some of them were measured.

In [ ]:
richter_prio_CHD_vars = ['chr1-114453712-A-G', 'chr1-119631728-CGGCGGGCGGCG-C',
       'chr1-148334937-C-T', 'chr1-154574841-CTAAA-C',
       'chr1-200029271-T-C', 'chr1-200275615-C-T', 'chr1-203650715-C-T',
       'chr1-205800738-C-A', 'chr1-210374057-TTACACA-T',
       'chr1-220818862-C-A', 'chr1-223749318-C-T', 'chr1-225882748-C-G',
       'chr1-234405976-C-T', 'chr1-248874196-G-A', 'chr1-26637027-C-T',
       'chr1-39639412-C-A', 'chr1-44213637-C-T', 'chr1-50182538-T-C',
       'chr1-61334943-G-T', 'chr1-61357295-G-C',
       'chr10-116627496-TGGGGCAGTCCC-T', 'chr10-129910016-C-T',
       'chr10-132577787-TCCTGGTCACTGGACAGCAG-T', 'chr10-14089430-G-T',
       'chr10-14837985-C-G', 'chr10-21241411-G-C', 'chr10-71191875-G-A',
       'chr10-71219722-A-T', 'chr10-78472427-G-T', 'chr10-93315574-G-A',
       'chr10-99450945-C-G', 'chr11-112777743-G-A', 'chr11-114216477-G-A',
       'chr11-118522576-G-C', 'chr11-119422669-G-C',
       'chr11-120934285-C-T', 'chr11-122038783-G-T',
       'chr11-126997860-T-C', 'chr11-128693763-A-C',
       'chr11-133136543-G-C', 'chr11-133930953-G-A', 'chr11-18791946-C-T',
       'chr11-34916100-C-T', 'chr11-60915559-C-G', 'chr11-61072184-C-T',
       'chr11-72984707-A-C', 'chr11-74721095-C-T', 'chr11-755546-T-C',
       'chr11-8081776-G-T', 'chr12-100573564-G-T', 'chr12-1130468-A-G',
       'chr12-19971568-C-T', 'chr12-20588376-G-T', 'chr12-26890943-T-C',
       'chr12-38892300-GACTTTACTCTGTAAAC-G', 'chr12-40017067-T-A',
       'chr12-42214329-T-A', 'chr12-51262920-G-A', 'chr12-52507256-A-G',
       'chr12-69066805-T-C', 'chr12-80814906-A-G', 'chr13-108629577-C-A',
       'chr13-30757255-C-T', 'chr13-39889643-G-T', 'chr13-52194461-C-T',
       'chr13-76853265-G-A', 'chr13-95698583-ACT-A',
       'chr14-101917508-C-T', 'chr14-68736858-C-A',
       'chr14-95262030-ACAGGCGCTGGGGCTGGGCCAG-A', 'chr14-96930147-A-G',
       'chr14-97218454-G-T', 'chr14-99424430-C-T', 'chr15-100132100-T-C',
       'chr15-64050611-G-C', 'chr15-65296074-G-T', 'chr15-68504536-G-A',
       'chr15-73926546-T-G', 'chr16-17194330-G-A', 'chr16-24344886-GA-G',
       'chr16-67720066-G-C', 'chr16-86929353-G-A', 'chr16-87723504-C-G',
       'chr16-87723512-C-T', 'chr16-9500150-A-C', 'chr17-32846370-C-G',
       'chr17-3530313-G-T', 'chr17-43661874-C-T', 'chr17-47709421-C-A',
       'chr17-64411582-TGCTGTGGGTGGGCATACGCAGTGGATGGGCATGTGCAGTGGATGGGCATGTGCAGTGGGTGGGCATACGCAGTGGGTGGGCATACACAGTGGGTGGGCATGTGCAGTGGGTGGGCATGTGCAGGGGGTGGGCATAC-T',
       'chr17-65074928-C-T', 'chr17-74296722-A-T', 'chr17-76390193-C-T',
       'chr18-13863105-G-A', 'chr18-46356381-G-T', 'chr18-48814105-A-G',
       'chr18-54389008-G-A', 'chr18-75639268-C-G', 'chr19-10086033-G-C',
       'chr19-18113114-G-A', 'chr19-18338050-G-A', 'chr19-35639201-C-T',
       'chr19-37078450-CG-C', 'chr19-46304160-G-GC', 'chr19-5567612-A-G',
       'chr2-110755091-G-A', 'chr2-150425290-G-A', 'chr2-166610768-T-C',
       'chr2-172480962-C-T', 'chr2-218909433-T-C', 'chr2-227785506-G-T',
       'chr2-232525907-C-T', 'chr2-25010233-A-G', 'chr2-26979970-T-A',
       'chr2-31001205-C-T', 'chr2-33555315-C-T', 'chr2-41972719-C-T',
       'chr2-47036188-G-C', 'chr2-62304829-G-T', 'chr2-70645924-C-G',
       'chr2-96089822-G-A', 'chr2-97774557-C-T', 'chr2-99389580-T-C',
       'chr20-32696005-T-C', 'chr20-52313584-C-T', 'chr20-59839643-C-A',
       'chr20-61425809-AGGCCCCGCCCAGGGCAGGATGGCCAATTGCAGTACATTCTCAAAGGCCCCGCCCAGGGCAGAATGGCCAATTGCAGGACGTTCTCGAG-A',
       'chr20-62652440-G-A', 'chr20-9507096-G-A', 'chr21-28737725-A-G',
       'chr21-38089151-G-A', 'chr21-39314372-C-G', 'chr21-42336262-G-A',
       'chr21-42955811-ACACCGCTGCGT-A', 'chr22-17492433-T-A',
       'chr22-29481081-G-A', 'chr3-11508823-A-G', 'chr3-115255152-G-T',
       'chr3-115523698-C-T', 'chr3-115761287-C-T', 'chr3-127049648-G-A',
       'chr3-134654141-C-T', 'chr3-136390775-C-T', 'chr3-138823908-A-T',
       'chr3-160772238-C-G', 'chr3-167167918-G-A', 'chr3-178361938-C-A',
       'chr3-180980799-A-ACC', 'chr3-184275946-C-G', 'chr3-193869984-T-C',
       'chr3-196081553-G-C', 'chr3-40238454-G-A', 'chr3-54600521-C-G',
       'chr3-59606954-C-G', 'chr3-69649962-A-G', 'chr3-70738097-GTGTT-G',
       'chr3-8741845-G-T', 'chr4-102299975-C-T', 'chr4-120049833-A-G',
       'chr4-121072350-G-GGCGGGAGGATGCCGCAGCGACCC', 'chr4-144654343-C-T',
       'chr4-169334368-C-T', 'chr4-183784017-G-C', 'chr4-185464347-A-G',
       'chr4-2393758-G-A', 'chr4-4107604-C-G', 'chr4-41935352-AG-A',
       'chr4-6257696-C-A', 'chr4-6641340-C-T', 'chr4-87108150-G-T',
       'chr5-114362083-A-G', 'chr5-130634-A-G', 'chr5-14901848-A-G',
       'chr5-150591206-T-G', 'chr5-153404963-G-A', 'chr5-1841170-T-C',
       'chr5-32628572-C-T', 'chr5-3681018-C-CA',
       'chr5-415471-GGTCGGGTGGGAGGCCTAGGGGCCGAGTCTCCCTGGTCGGGTGGGAGGCCTAGGGGCCGA-G',
       'chr5-65184846-C-T', 'chr5-88892842-G-A',
       'chr5-90595612-GCTGACCCCCCCACCTCCCTCCCGGACGGGGCGGCTGGCCGGGCGGGGAA-G',
       'chr6-111087293-C-G', 'chr6-111382839-A-G', 'chr6-13795361-G-C',
       'chr6-16286311-T-A', 'chr6-163132634-C-T', 'chr6-166863036-G-C',
       'chr6-21407899-C-G', 'chr6-24872997-G-A', 'chr6-25974140-A-C',
       'chr6-27230575-C-CCCACGCGGACACCTGT', 'chr6-2940385-C-T',
       'chr6-4647705-A-G', 'chr6-46809067-G-A', 'chr6-56833586-A-G',
       'chr6-69816780-C-T', 'chr6-77464378-C-T', 'chr6-81382963-G-C',
       'chr6-81752256-G-C', 'chr7-101322293-GT-G', 'chr7-12475940-G-A',
       'chr7-134480919-T-C', 'chr7-143484974-G-C', 'chr7-151210653-G-A',
       'chr7-19442368-AC-A',
       'chr7-257782-GGGGTGCTGGAGATGGGCTGGGTGGACCCACTGCCCGGGGTGCTGGAGATGGGCAGGGTGGACCCACTGCCTGGGGTGCTGGAGATGGGGCTGGGTGGACCCACTGCCCAGGATGCTGGAGATGGGCTGGGTGGACACACTGCCTGGGGTGCTGGAGATGCCCGGGGTGGACCCACTGCCTGA-G',
       'chr7-39609853-G-A', 'chr7-6059060-G-C', 'chr7-74580511-C-T',
       'chr7-8287350-T-C', 'chr7-87876606-T-C', 'chr8-106795479-G-T',
       'chr8-108142323-C-T', 'chr8-11742663-C-A', 'chr8-122942170-C-G',
       'chr8-127854377-G-T', 'chr8-133406700-C-T', 'chr8-133725408-C-T',
       'chr8-133898135-G-A', 'chr8-140047232-C-T', 'chr8-141801526-GGA-G',
       'chr8-8247991-C-T', 'chr8-90991301-G-A', 'chr8-95225792-G-T',
       'chr9-113455521-A-G', 'chr9-121558170-C-T', 'chr9-128340088-G-T',
       'chr9-14400787-C-T', 'chr9-3980382-C-T', 'chr9-79768314-T-G',
       'chr9-86589149-G-C', 'chrX-103330714-T-C']

In [ ]:
sdr7_mast_a = pd.read_csv(SDR007_FILE_PATH / "SDR007A_phased_MAST_cell_annot_day_complement.csv")
sdr7_mast_b = pd.read_csv(SDR007_FILE_PATH / "SDR007B_phased_MAST_cell_annot_day_complement.csv")

raw_sdr7_mast = pd.concat([sdr7_mast_a, sdr7_mast_b])
raw_sdr7_mast = raw_sdr7_mast[raw_sdr7_mast.Gene != "DUMMY-FEATURE-FOR-SEURAT"]

In [ ]:
HET_mask = raw_sdr7_mast["ID_Geno_Var1_HET"].apply(
    lambda x: isinstance(x, str) and any(v in x for v in richter_prio_CHD_vars)
)

ALT_mask = raw_sdr7_mast["ID_Geno_Var1_ALT"].apply(
    lambda x: isinstance(x, str) and any(v in x for v in richter_prio_CHD_vars)
)

raw_sdr7_mast["is_prio"] = HET_mask | ALT_mask

In [ ]:
raw_sdr7_mast["-log10_pval"] = -np.log10(raw_sdr7_mast["p_val_adj"])
raw_sdr7_mast["abs_log2FC"] = raw_sdr7_mast["avg_log2FC"].abs() 

In [ ]:
raw_sdr7_mast["variant_id"] = raw_sdr7_mast.apply(
    lambda row: (
        f"{row['ID_Geno_Var1_HET']}|{row['ID_Geno_Var1_ALT']}"
        if pd.notna(row["ID_Geno_Var1_HET"]) and pd.notna(row["ID_Geno_Var1_ALT"])
        else row["ID_Geno_Var1_HET"]
        if pd.notna(row["ID_Geno_Var1_HET"])
        else row["ID_Geno_Var1_ALT"]
    ),
    axis=1,
)

In [ ]:
gene_counts = (
    raw_sdr7_mast.loc[raw_sdr7_mast["p_val_adj"] < 0.05]
    .groupby(["variant_id", "is_prio"])["Gene"]
    .nunique()
    .reset_index(name="n_sig_genes")
)

gene_counts.groupby("is_prio").n_sig_genes.mean()

In [ ]:
box_grid(gene_counts, x_col = "is_prio", y_cols = "n_sig_genes", alternative = "greater")